# Aprendizaje Automático — Aprendizaje No Supervisado
## Entrega 2 - Exploración Inicial del Dataset
**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial**  
Politécnico Malvinas Argentinas

---

## 🎯 Objetivo
El objetivo de este proceso es conocer los datos antes de aplicar
cualquier técnica de Aprendizaje Automático. Esta exploración inicial me permite:

- **Entender la estructura del dataset:** cantidad de registros, variables y tipos de datos.
- **Identificar valores faltantes o códigos de no respuesta** (códigos 9, 99, 999 según el diccionario de la ENPreCoSP 2011) que deberán tratarse en el preprocesamiento.
- **Conocer la distribución de las variables** seleccionadas para el clustering.
- **Verificar los subconjuntos de trabajo:** jóvenes de 16 a 24 años a nivel nacional (6.592 registros) y el subconjunto de Tierra del Fuego (265 registros, PRVNC = 94).
- **Describir el perfil de consumo** de la población de interés mediante las variables de prevalencia del último año (P1A_BA, P1A_TA, P1A_MA, P1A_CO, P1A_PB, P1A_TR).

Esta exploración es la base para las decisiones de preprocesamiento y modelado que podría desarrollar en la siguiente etapa.

**Dataset:** `Base Usuario ENPreCoSP-2011.txt`

---
> Nota: Esta exploración inicial la realicé con en único objetivo de conocer la estructura del dataset y poder elaborar el informe de la segunda entrega parcial. Por el momento, no forma parte de mi entrega final. Lo aclaro, por si existen desprolijidades o falta de información y visualizaciones.

In [7]:
# Exploración del Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
PALETTE = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}
ORDER   = ['Low', 'Medium', 'High']

# ---- CARGA DEL DATASET ----
df = pd.read_csv('/content/drive/MyDrive/Aprendizaje Automático/Parcial/Archivos Parcial/Base Usuario ENPreCoSP-2011.txt',
                 sep='|',
                 encoding='latin-1',
                 index_col=False)

# ---- DIMENSIONES GENERALES ----
print("=== DIMENSIONES DEL DATASET COMPLETO ===")
print(f"Filas (registros): {df.shape[0]}")
print(f"Columnas (variables): {df.shape[1]}")

# ---- TIPOS DE DATOS ----
print("\n=== TIPOS DE DATOS ===")
print(df.dtypes.value_counts())

# ---- FILTRO JÓVENES 16-24 AÑOS ----
jovenes = df[df['GRUPEDAD'] == 2]
print(f"\n=== SUBCONJUNTO JÓVENES 16-24 AÑOS ===")
print(f"Registros nacionales: {len(jovenes)}")

# ---- FILTRO TIERRA DEL FUEGO ----
tdf = jovenes[jovenes['PRVNC'] == 94]
print(f"Registros Tierra del Fuego: {len(tdf)}")

# ---- VARIABLES DEL PROYECTO ----
vars_proyecto = ['BHCH04','BHCH05','NIVINSTR','CONDACT','TIPO_H',
                 'NBI_TOTAL','RANGOING','BISG01','BISG04','BIAC01',
                 'BIAC02','BIAC03','BIAC04','REGION','POB_URB','PRVNC']

print("\n=== VARIABLES PREDICTORAS ===")
print(jovenes[vars_proyecto].describe())

# ---- VALORES NULOS ----
print("\n=== VALORES NULOS POR VARIABLE ===")
print(jovenes[vars_proyecto].isnull().sum())

# ---- VARIABLES DE CONSUMO ----
vars_consumo = ['P1A_BA','P1A_TA','P1A_MA','P1A_CO','P1A_PB','P1A_TR']
print("\n=== VARIABLES DE CONSUMO (último año) ===")
for v in vars_consumo:
    si = (jovenes[v] == 1).sum()
    total = jovenes[v].notna().sum()
    pct = si/total*100
    print(f"{v}: {si} consumidores ({pct:.1f}%)")

=== DIMENSIONES DEL DATASET COMPLETO ===
Filas (registros): 34343
Columnas (variables): 291

=== TIPOS DE DATOS ===
float64    166
int64      125
Name: count, dtype: int64

=== SUBCONJUNTO JÓVENES 16-24 AÑOS ===
Registros nacionales: 6592
Registros Tierra del Fuego: 265

=== VARIABLES PREDICTORAS ===
            BHCH04       BHCH05     NIVINSTR      CONDACT       TIPO_H  \
count  6592.000000  6592.000000  6592.000000  6592.000000  6592.000000   
mean      1.519266    20.125000     4.552336     2.029430     4.811742   
std       0.499667     2.543179     1.182481     0.965046     1.770790   
min       1.000000    16.000000     1.000000     1.000000     1.000000   
25%       1.000000    18.000000     4.000000     1.000000     4.000000   
50%       2.000000    20.000000     4.000000     2.000000     4.000000   
75%       2.000000    22.000000     6.000000     3.000000     6.000000   
max       2.000000    24.000000     8.000000     3.000000     8.000000   

         NBI_TOTAL     RANGOING

In [8]:
# PERFIL DEL SUBCONJUNTO JÓVENES 16-24 AÑOS

print("\n" + "="*60)
print("PERFIL DEL SUBCONJUNTO DE TRABAJO (16-24 AÑOS)")
print("="*60)

total = len(jovenes)

# ---- SEXO ----
varones = (jovenes['BHCH04'] == 1).sum()
mujeres = (jovenes['BHCH04'] == 2).sum()

pct_varones = varones / total * 100
pct_mujeres = mujeres / total * 100

print("\nSexo")
print(f"Varón: {varones:,} ({pct_varones:.1f}%)")
print(f"Mujer: {mujeres:,} ({pct_mujeres:.1f}%)")

# ---- EDAD ----
edad_prom = jovenes['BHCH05'].mean()
edad_min = jovenes['BHCH05'].min()
edad_max = jovenes['BHCH05'].max()

print("\nEdad")
print(f"Promedio: {edad_prom:.1f} años")
print(f"Mínima: {edad_min}")
print(f"Máxima: {edad_max}")

# ---- CONSUMO ÚLTIMO AÑO ----
consumos = {
    'Alcohol (último año)': 'P1A_BA',
    'Tabaco (último año)': 'P1A_TA',
    'Marihuana (último año)': 'P1A_MA',
    'Cocaína (último año)': 'P1A_CO',
    'Pasta base (último año)': 'P1A_PB',
    'Tranquilizantes (último año)': 'P1A_TR'
}

print("\nConsumo de sustancias (último año)")
print("-"*60)

for nombre, variable in consumos.items():
    consumidores = (jovenes[variable] == 1).sum()
    porcentaje = consumidores / total * 100

    print(f"{nombre}: {consumidores:,} consumidores ({porcentaje:.1f}%)")


PERFIL DEL SUBCONJUNTO DE TRABAJO (16-24 AÑOS)

Sexo
Varón: 3,169 (48.1%)
Mujer: 3,423 (51.9%)

Edad
Promedio: 20.1 años
Mínima: 16
Máxima: 24

Consumo de sustancias (último año)
------------------------------------------------------------
Alcohol (último año): 4,444 consumidores (67.4%)
Tabaco (último año): 2,139 consumidores (32.4%)
Marihuana (último año): 283 consumidores (4.3%)
Cocaína (último año): 61 consumidores (0.9%)
Pasta base (último año): 4 consumidores (0.1%)
Tranquilizantes (último año): 74 consumidores (1.1%)


In [9]:
# VARIABLES SELECCIONADAS PARA EL PROYECTO

descripcion = {
    'BHCH04': 'Sexo',
    'BHCH05': 'Edad en años cumplidos',
    'NIVINSTR': 'Nivel de instrucción',
    'CONDACT': 'Condición de actividad',
    'TIPO_H': 'Tipo de hogar',
    'NBI_TOTAL': 'Necesidades Básicas Insatisfechas',
    'RANGOING': 'Rango de ingreso del hogar',
    'BISG01': 'Autopercepción de salud general',
    'BISG04': 'Visita profesional de salud mental',
    'BIAC01': 'Conoce consumidores cercanos',
    'BIAC02': 'Cantidad de consumidores cercanos',
    'BIAC03': 'Curiosidad por probar drogas',
    'BIAC04': 'Posibilidad de acceso a drogas',
    'REGION': 'Región estadística del país',
    'POB_URB': 'Tamaño de aglomerado urbano',
    'PRVNC': 'Provincia de residencia'
}

categoria = {
    'BHCH04': 'Sociodemográfica',
    'BHCH05': 'Sociodemográfica',
    'NIVINSTR': 'Sociodemográfica',
    'CONDACT': 'Sociodemográfica',

    'TIPO_H': 'Contexto familiar',
    'NBI_TOTAL': 'Contexto familiar',
    'RANGOING': 'Contexto familiar',

    'BISG01': 'Salud y entorno',
    'BISG04': 'Salud y entorno',
    'BIAC01': 'Salud y entorno',
    'BIAC02': 'Salud y entorno',
    'BIAC03': 'Salud y entorno',
    'BIAC04': 'Salud y entorno',

    'REGION': 'Territorial',
    'POB_URB': 'Territorial',
    'PRVNC': 'Territorial'
}

# Construcción de la tabla resumen
tabla_variables = pd.DataFrame({
    'Variable': vars_proyecto,
    'Descripción': [descripcion[v] for v in vars_proyecto],
    'Categoría': [categoria[v] for v in vars_proyecto],
    'Valores únicos': [jovenes[v].nunique(dropna=True) for v in vars_proyecto],
    'Nulos/Ns': [jovenes[v].isnull().sum() for v in vars_proyecto]
})

print("\n=== VARIABLES SELECCIONADAS PARA EL PROYECTO ===")
print(tabla_variables)

# Opcional: mostrar completa sin cortes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(tabla_variables)


=== VARIABLES SELECCIONADAS PARA EL PROYECTO ===
     Variable                         Descripción          Categoría  \
0      BHCH04                                Sexo   Sociodemográfica   
1      BHCH05              Edad en años cumplidos   Sociodemográfica   
2    NIVINSTR                Nivel de instrucción   Sociodemográfica   
3     CONDACT              Condición de actividad   Sociodemográfica   
4      TIPO_H                       Tipo de hogar  Contexto familiar   
5   NBI_TOTAL   Necesidades Básicas Insatisfechas  Contexto familiar   
6    RANGOING          Rango de ingreso del hogar  Contexto familiar   
7      BISG01     Autopercepción de salud general    Salud y entorno   
8      BISG04  Visita profesional de salud mental    Salud y entorno   
9      BIAC01        Conoce consumidores cercanos    Salud y entorno   
10     BIAC02   Cantidad de consumidores cercanos    Salud y entorno   
11     BIAC03        Curiosidad por probar drogas    Salud y entorno   
12     BIAC04 

In [10]:
# VARIABLES DE CONSUMO (ÚLTIMO AÑO)

variables_consumo = {
    'P1A_BA': 'Bebidas alcohólicas',
    'P1A_TA': 'Tabaco',
    'P1A_MA': 'Marihuana',
    'P1A_CO': 'Cocaína',
    'P1A_PB': 'Pasta base',
    'P1A_TR': 'Tranquilizantes sin receta'
}

tabla_consumo = []

for var, sustancia in variables_consumo.items():

    consumidores = (jovenes[var] == 1).sum()

    # porcentaje sobre el total de jóvenes
    porcentaje = consumidores / len(jovenes) * 100

    tabla_consumo.append([
        var,
        sustancia,
        consumidores,
        round(porcentaje, 1)
    ])

tabla_consumo = pd.DataFrame(
    tabla_consumo,
    columns=[
        'Variable',
        'Sustancia',
        'Consumidores (último año)',
        'Porcentaje (%)'
    ]
)

print("\n=== VARIABLES DE CONSUMO ===")
print(tabla_consumo)


=== VARIABLES DE CONSUMO ===
  Variable                   Sustancia  Consumidores (último año)  \
0   P1A_BA         Bebidas alcohólicas                       4444   
1   P1A_TA                      Tabaco                       2139   
2   P1A_MA                   Marihuana                        283   
3   P1A_CO                     Cocaína                         61   
4   P1A_PB                  Pasta base                          4   
5   P1A_TR  Tranquilizantes sin receta                         74   

   Porcentaje (%)  
0            67.4  
1            32.4  
2             4.3  
3             0.9  
4             0.1  
5             1.1  


In [11]:
print("\nVARIABLES DE CONSUMO (ANÁLISIS DESCRIPTIVO)")
print("-"*80)

for _, fila in tabla_consumo.iterrows():
    print(
        f"{fila['Variable']:8} | "
        f"{fila['Sustancia']:30} | "
        f"{fila['Consumidores (último año)']:5,d} consumidores | "
        f"{fila['Porcentaje (%)']:.1f}%"
    )


VARIABLES DE CONSUMO (ANÁLISIS DESCRIPTIVO)
--------------------------------------------------------------------------------
P1A_BA   | Bebidas alcohólicas            | 4,444 consumidores | 67.4%
P1A_TA   | Tabaco                         | 2,139 consumidores | 32.4%
P1A_MA   | Marihuana                      |   283 consumidores | 4.3%
P1A_CO   | Cocaína                        |    61 consumidores | 0.9%
P1A_PB   | Pasta base                     |     4 consumidores | 0.1%
P1A_TR   | Tranquilizantes sin receta     |    74 consumidores | 1.1%


In [12]:
tabla_consumo.style.format({
    'Consumidores (último año)': '{:,.0f}',
    'Porcentaje (%)': '{:.1f}%'
})

,Variable,Sustancia,Consumidores (último año),Porcentaje (%)
0,P1A_BA,Bebidas alcohólicas,"4,444",67.4%
1,P1A_TA,Tabaco,"2,139",32.4%
2,P1A_MA,Marihuana,283,4.3%
3,P1A_CO,Cocaína,61,0.9%
4,P1A_PB,Pasta base,4,0.1%
5,P1A_TR,Tranquilizantes sin receta,74,1.1%
